# PRACTICE 1: CLASSIFYING SPAM EMAILS
## PART 1: KHÁM PHÁ DỮ LIỆU & LÀM SẠCH VĂN BẢN (DATA CLEANING & EDA)

```text
[CEAS_08.csv] ──> Mục 1 (EDA Nền tảng: Cấu trúc, Chất lượng & Đơn biến)
              ──> Mục 2 (EDA Chuyên sâu: Đa biến, Ngoại lai & Đúc kết định hướng)
              ──> Mục 3 (Làm sạch dữ liệu: Khử trùng, điền missing, clean_text) ──> [clean_emails.csv]
```


---
## Mục 1: Khám phá dữ liệu tổng quan — Cấu trúc, Chất lượng & Đơn biến (Overview EDA)


In [ ]:
# ==============================================================================
# 1.1 KHAI BÁO THƯ VIỆN & CẤU HÌNH RANDOM SEED
# ==============================================================================
import os
import sys
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', palette='muted')

DATA_RAW_PATH = '../data/raw/CEAS_08.csv'
print(f'✅ Môi trường sẵn sàng với Python {sys.version.split()[0]} | RANDOM_SEED = {RANDOM_SEED}')

In [ ]:
# ==============================================================================
# 1.2 KHẢO SÁT CẤU TRÚC & ĐÁNH GIÁ CHẤT LƯỢNG DỮ LIỆU (EDA 1 & 2)
# ==============================================================================
df = pd.read_csv(DATA_RAW_PATH)
print(f"📊 Kích thước dữ liệu gốc: {df.shape[0]:,} dòng, {df.shape[1]} cột")
print("-" * 60)
print("Thông tin kiểu dữ liệu và bộ nhớ:")
df.info()

print("\nNăm dòng đầu tiên của dữ liệu:")
display(df.head(5))

print("\nThống kê mô tả các biến số:")
display(df.describe())

# Đánh giá giá trị khuyết thiếu (Missing values)
missing_count = df.isnull().sum()
missing_percent = (missing_count / len(df)) * 100
missing_summary = pd.DataFrame({
    'Số lượng thiếu': missing_count,
    'Tỷ lệ thiếu (%)': missing_percent.round(4)
})
print("\nThống kê giá trị khuyết thiếu trên các cột:")
display(missing_summary[missing_summary['Số lượng thiếu'] > 0])

# Đánh giá bản ghi trùng lặp (Duplicate records)
duplicate_count = df.duplicated().sum()
print(f"\n• Số dòng trùng lặp hoàn toàn: {duplicate_count:,} dòng ({(duplicate_count/len(df))*100:.2f}%)")


In [ ]:
# ==============================================================================
# 1.3 PHÂN TÍCH ĐƠN BIẾN TOÀN DIỆN 4 THUỘC TÍNH (EDA 3)
# ==============================================================================
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Biến mục tiêu: label (0: Ham, 1: Spam)
label_counts = df['label'].value_counts()
sns.countplot(data=df, x='label', palette=['#3498db', '#e74c3c'], ax=axes[0, 0])
axes[0, 0].set_title(f"1. Biến mục tiêu: Ham ({label_counts.get(0, 0):,}) vs Spam ({label_counts.get(1, 0):,})")
axes[0, 0].set_xlabel('Nhãn (0: Ham / Hợp lệ, 1: Spam / Thư rác)')
axes[0, 0].set_ylabel('Số lượng email')
for p in axes[0, 0].patches:
    axes[0, 0].annotate(f"{int(p.get_height()):,} ({p.get_height()/len(df)*100:.1f}%)",
                         (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                         ha='center', va='center', color='white', fontweight='bold', fontsize=11)

# 2. Biến số độc lập: urls (Có/Không link URL)
sns.countplot(data=df, x='urls', palette='viridis', ax=axes[0, 1])
axes[0, 1].set_title('2. Biến số URLs: Tỷ lệ email có chứa liên kết')
axes[0, 1].set_xlabel('Số lượng liên kết URL (0: Không, 1+: Có)')
axes[0, 1].set_ylabel('Số lượng email')

# 3. Biến văn bản: subject length (Độ dài tiêu đề)
df['subject_len'] = df['subject'].fillna('').astype(str).apply(len)
sns.histplot(df['subject_len'], bins=50, kde=True, color='#2ecc71', ax=axes[1, 0])
axes[1, 0].set_title('3. Phân phối độ dài tiêu đề (Subject Length)')
axes[1, 0].set_xlabel('Số ký tự trong tiêu đề')
axes[1, 0].set_ylabel('Mật độ / Số lượng')
axes[1, 0].set_xlim(0, 200)

# 4. Biến danh mục: sender domain (Tên miền người gửi)
df['sender_domain'] = df['sender'].astype(str).apply(lambda x: x.split('@')[-1].lower() if '@' in x else 'unknown')
top_domains = df['sender_domain'].value_counts().head(10)
top_domains.plot(kind='bar', color='#f39c12', ax=axes[1, 1])
axes[1, 1].set_title('4. Top 10 tên miền người gửi phổ biến nhất')
axes[1, 1].set_xlabel('Tên miền (Domain)')
axes[1, 1].set_ylabel('Số lượng email')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


---
## Mục 2: Khám phá dữ liệu chuyên sâu — Đa biến, Ngoại lai & Đúc kết định hướng (In-depth EDA)


In [ ]:
# Viết code phân tích đa biến, ngoại lai IQR độ dài body, Boxplot so sánh theo nhãn và tương quan URL tại đây


In [ ]:
# Viết code hiển thị Bảng đúc kết kết luận EDA Bước 5 (Bằng chứng -> Tác động -> Định hướng hành động) tại đây


---
## Mục 3: Tiền xử lý dữ liệu & Làm sạch văn bản (Data Cleaning)


In [ ]:
# Viết code khử trùng lặp và điền chuỗi rỗng cho 28 email khuyết subject tại đây


In [ ]:
# Viết hàm clean_text (loại bỏ HTML, URL, ký tự đặc biệt, lowercase, stopwords) và gộp subject + body thành text_clean tại đây


In [ ]:
# Viết code xuất tệp dữ liệu sạch vào '../data/processed/clean_emails.csv' tại đây
